# Branch 3 BEV U-Net Colab Retrain — split CPU/GPU staged workflow

Uses `gcloud storage` for all durable batch transfers. CPU stages generate BEV labels and push labeled session folders; GPU stages pull labeled batches and train the U-Net without regenerating labels.


In [ ]:
# Google Drive is no longer required for this notebook.
# Durable inputs/outputs are expected under the configured GCS bucket.


Mounted at /content/drive


In [ ]:
import shutil
from pathlib import Path

shutil.rmtree(Path('/content/work/code/data/_rolling_branch1_rdra_batches/external_validation'), ignore_errors=True)

In [ ]:
# 1. Update the gcloud CLI (Google Cloud SDK) to ensure the latest 'gcloud storage' features are available
!echo "deb [signed-by=/usr/share/keyrings/cloud.google.gpg] https://packages.cloud.google.com/apt cloud-sdk main" | sudo tee -a /etc/apt/sources.list.d/google-cloud-sdk.list
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key --keyring /usr/share/keyrings/cloud.google.gpg add -
!sudo apt-get update && sudo apt-get install -y google-cloud-cli

!gcloud --version

from google.colab import auth
auth.authenticate_user()

# Match the Branch 1 retrain notebook GCP/GCS setup.
PROJECT_ID = "fluent-webbing-496616-u8"
BUCKET_NAME = "miamioh-resa-data"
GCS_MOUNT_POINT = "/content/gcs"

!gcloud config set project {PROJECT_ID}

# Tune `gcloud storage` for parallel multi-file transfers.
!gcloud config set storage/process_count 4
!gcloud config set storage/thread_count 16

# Install gcsfuse only when it is not already available.
!if ! command -v gcsfuse >/dev/null 2>&1; then \
  echo "Installing gcsfuse..."; \
  export GCSFUSE_REPO=gcsfuse-`lsb_release -c -s`; \
  echo "deb https://packages.cloud.google.com/apt $GCSFUSE_REPO main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list; \
  curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -; \
  sudo apt-get update; \
  sudo apt-get install -y gcsfuse; \
else \
  echo "gcsfuse already installed."; \
fi

!mkdir -p {GCS_MOUNT_POINT}
!umount {GCS_MOUNT_POINT} 2>/dev/null || true
!gcsfuse --implicit-dirs \
   --file-cache-max-size-mb=40000 \
   {BUCKET_NAME} {GCS_MOUNT_POINT}

print(f"\nSuccessfully mounted {BUCKET_NAME} to {GCS_MOUNT_POINT}")


In [ ]:
%%bash
set -euo pipefail

python3 -m pip install --upgrade pip
python3 -m pip install pandas numpy tqdm scipy scikit-learn

mkdir -p /content/work
mkdir -p /content/work/code
mkdir -p /home/hullumdr/prof
ln -sfn /content/work /home/hullumdr/prof/io

gcloud storage rsync --recursive   --exclude 'LLM_ML/data/.*|data/.*|outputs/.*|__pycache__/.*|\.ipynb_checkpoints/.*'   gs://miamioh-resa-data/CapstoneData/code_v2/RESA_mmWave/   /content/work/code/

echo 'setup complete'


In [ ]:
import csv
import json
import os
import random
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

import numpy as np
import pandas as pd

BUCKET_NAME = 'miamioh-resa-data'
GCS_ROOT_URI = f'gs://{BUCKET_NAME}/CapstoneData'
GCS_CATALOG_URI = f'{GCS_ROOT_URI}/curated/manifests/session_catalog.csv'
CURATED_ROOT_URI = f'{GCS_ROOT_URI}/curated/sessions'
BEV_STAGE_ROOT_URI = f'{GCS_ROOT_URI}/intermediate/bev_labels'
PUBLISHED_BRANCH3_ROOT_URI = f'{GCS_ROOT_URI}/published/branch3/checkpoints'
PUBLISHED_RUNTIME_UNET_URI = f'{PUBLISHED_BRANCH3_ROOT_URI}/unet_best_model.pt'

WORK_ROOT = '/content/work'
ROOT = Path(WORK_ROOT)
CODE = ROOT / 'code'
PY = Path(sys.executable)
CATALOG_LOCAL_PATH = ROOT / '_session_catalog' / 'session_catalog.csv'
ROLLING_ROOT = ROOT / '_rolling_branch3_unet_batches'
ROLLING_PROCESSING_ROOT = ROLLING_ROOT / 'processing'
ROLLING_DATASET_ROOT = ROLLING_ROOT / 'datasets'
ROLLING_VAL_ROOT = ROLLING_ROOT / 'external_validation'
FULL_ROOT = ROLLING_ROOT / 'full'
TRAIN_OUT = ROOT / '_train_output' / 'branch3_unet_curated_split_v1'
DRIVE_TRAIN_OUT_URI = f'{PUBLISHED_BRANCH3_ROOT_URI}/{TRAIN_OUT.name}'
RUNTIME_UNET_PT = ROOT / '_runtime' / 'unet_best_model.pt'

TRAIN_QUALITY_TIERS = tuple(globals().get('TRAIN_QUALITY_TIERS', ('gold',)))
VAL_QUALITY_TIERS = tuple(globals().get('VAL_QUALITY_TIERS', ('gold',)))
TRAIN_SPLITS = tuple(globals().get('TRAIN_SPLITS', ('train',)))
EXTERNAL_VAL_SPLITS = tuple(globals().get('EXTERNAL_VAL_SPLITS', ('external_val', 'holdout', 'val', 'test')))
TRAIN_DATASET_IDS = tuple(globals().get('TRAIN_DATASET_IDS', ()))
EXTERNAL_VAL_DATASET_IDS = tuple(globals().get('EXTERNAL_VAL_DATASET_IDS', ()))
PROMOTE_BEST_TO_RUNTIME = bool(globals().get('PROMOTE_BEST_TO_RUNTIME', True))

UNET_TRAIN_IN_BATCHES = True
UNET_SESSION_DOWNLOAD_BATCH_SIZE = 20
UNET_STREAMING_PASSES = 20
UNET_EPOCHS_PER_SESSION_BATCH = 1
UNET_FULL_EPOCHS = 50

UNET_BATCH_SIZE = 16
UNET_LR = 3.14e-4
UNET_WEIGHT_DECAY = 1e-4
UNET_BASE_CH = 32
UNET_PATIENCE = 15
UNET_NUM_WORKERS = 2
UNET_SEED = 42
UNET_BEST_METRIC = 'ext_val_iou'

ENV = os.environ.copy()
ENV['PYTHONPATH'] = f'{ROOT}:{CODE}' + (':' + ENV['PYTHONPATH'] if ENV.get('PYTHONPATH') else '')
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

from tools.session_selector import load_catalog, select_sessions, sync_catalog_from_gcs


def first_existing(*paths):
    for path in paths:
        p = Path(path)
        if p.exists():
            return p
    raise FileNotFoundError('Missing required path. Checked: ' + ', '.join(str(p) for p in paths))


BRANCH3_LABEL_BEV = first_existing(CODE / 'branch3' / 'branch3_label_bev.py')
BRANCH3_UNET_MODEL = first_existing(CODE / 'branch3' / 'branch3_unet.py')
TRAIN_SCRIPT = first_existing(CODE / 'branch3' / 'train_unet.py')


def run(cmd, cwd=ROOT):
    cmd = [str(x) for x in cmd]
    print()
    print('$ ' + ' '.join(cmd), flush=True)
    print('cwd:', cwd, flush=True)
    r = subprocess.run(cmd, cwd=str(cwd), env=ENV, check=True, capture_output=True, text=True)
    if r.stdout:
        print()
        print('--- STDOUT ---')
        print(r.stdout)
    if r.stderr:
        print()
        print('--- STDERR ---')
        print(r.stderr)
    return r


def run_shell(script, cwd=ROOT):
    print()
    print('$ bash -lc', script[:180].replace('\n', ' ; '), '...', flush=True)
    subprocess.run(['bash', '-lc', script], cwd=str(cwd), env=ENV, check=True)


def path_to_gs_uri(path):
    s = str(path)
    if s.startswith('gs://'):
        return s.rstrip('/')
    prefix = '/content/gcs/'
    if s.startswith(prefix):
        return f"gs://{BUCKET_NAME}/" + s[len(prefix):].strip('/')
    return s.rstrip('/')


def gcloud_storage_rsync(src, dst, *, delete=False):
    src_uri = path_to_gs_uri(src)
    dst_uri = path_to_gs_uri(dst)
    cmd = ['gcloud', 'storage', 'rsync', '--recursive']
    if delete:
        cmd.append('--delete-unmatched-destination-objects')
    cmd.extend([src_uri, dst_uri])
    return run(cmd)


def gcloud_storage_cp(src, dst):
    return run(['gcloud', 'storage', 'cp', path_to_gs_uri(src), path_to_gs_uri(dst)])


def gcloud_storage_ls(pattern):
    cmd = ['gcloud', 'storage', 'ls', pattern]
    print()
    print('$ ' + ' '.join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=str(ROOT), env=ENV, capture_output=True, text=True)
    if r.returncode != 0:
        if r.stderr:
            print(r.stderr)
        return []
    return [line.strip() for line in r.stdout.splitlines() if line.strip()]


def parse_csv_list(raw):
    return tuple(part.strip() for part in str(raw).split(',') if part and part.strip())


def sync_live_catalog():
    CATALOG_LOCAL_PATH.parent.mkdir(parents=True, exist_ok=True)
    return sync_catalog_from_gcs(CATALOG_LOCAL_PATH, GCS_CATALOG_URI)


def curated_records(*, quality_tiers, include_splits, dataset_ids=()):
    df = load_catalog(sync_live_catalog())
    rows = select_sessions(df, quality_tier=quality_tiers, split_exclude=('none', ''))
    include_splits = set(parse_csv_list(include_splits) if isinstance(include_splits, str) else include_splits)
    if include_splits:
        rows = rows[rows['split'].astype(str).isin(include_splits)]
    dataset_ids = set(parse_csv_list(dataset_ids) if isinstance(dataset_ids, str) else dataset_ids)
    if dataset_ids:
        rows = rows[rows['dataset_id'].astype(str).isin(dataset_ids)]
    rows = rows.sort_values(['dataset_id', 'session_id']).reset_index(drop=True)
    if rows.empty:
        raise RuntimeError('No curated sessions matched the current Branch 3 filters.')
    return [
        (str(row.dataset_id), str(row.session_id), str(row.processed_path), str(row.split))
        for row in rows.itertuples(index=False)
    ]


def validation_session_records():
    return curated_records(
        quality_tiers=VAL_QUALITY_TIERS,
        include_splits=EXTERNAL_VAL_SPLITS,
        dataset_ids=EXTERNAL_VAL_DATASET_IDS,
    )


def session_records():
    return curated_records(
        quality_tiers=TRAIN_QUALITY_TIERS,
        include_splits=TRAIN_SPLITS,
        dataset_ids=TRAIN_DATASET_IDS,
    )


def ensure_expanded_sessions_available():
    train_records = session_records()
    val_records = validation_session_records()
    print(f'Curated train sessions available: {len(train_records)}')
    print(f'Curated external-val sessions available: {len(val_records)}')


def chunks(items, size):
    for i in range(0, len(items), int(size)):
        yield i // int(size) + 1, items[i:i + int(size)]


def clear_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def clear_rolling_training_dirs():
    for path in [ROLLING_PROCESSING_ROOT, ROLLING_DATASET_ROOT]:
        if path.exists():
            shutil.rmtree(path)
    ROLLING_PROCESSING_ROOT.mkdir(parents=True, exist_ok=True)
    ROLLING_DATASET_ROOT.mkdir(parents=True, exist_ok=True)


def extract_session_archive(archive_path, dst_root, expected_session_name):
    dst_root = Path(dst_root)
    tmp_root = dst_root / '_extract_tmp' / expected_session_name
    shutil.rmtree(tmp_root, ignore_errors=True)
    tmp_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, 'r:gz') as tf:
        base = tmp_root.resolve()
        for member in tf.getmembers():
            target = (tmp_root / member.name).resolve()
            if not str(target).startswith(str(base)):
                raise RuntimeError(f'Unsafe archive path in {archive_path}: {member.name}')
        tf.extractall(tmp_root)
    direct = tmp_root / expected_session_name
    final_session = dst_root / expected_session_name
    shutil.rmtree(final_session, ignore_errors=True)
    if direct.exists() and direct.is_dir():
        shutil.move(str(direct), str(final_session))
    else:
        final_session.mkdir(parents=True, exist_ok=True)
        for item in sorted(tmp_root.iterdir()):
            shutil.move(str(item), str(final_session / item.name))
    shutil.rmtree(dst_root / '_extract_tmp', ignore_errors=True)
    return final_session


def download_session_records(records, processing_root):
    processing_root = Path(processing_root)
    processing_root.mkdir(parents=True, exist_ok=True)
    archive_root = processing_root / '_archives'
    archive_root.mkdir(parents=True, exist_ok=True)
    counts = {}
    dataset_roots = {}
    for dataset_id, session_name, source_uri, _split in records:
        dataset_root = processing_root / dataset_id
        dataset_root.mkdir(parents=True, exist_ok=True)
        archive_path = archive_root / f'{session_name}.tar.gz'
        print(f'Downloading curated session via gcloud: {source_uri} -> {archive_path}')
        gcloud_storage_cp(source_uri, archive_path)
        extract_session_archive(archive_path, dataset_root, session_name)
        counts[dataset_id] = counts.get(dataset_id, 0) + 1
        dataset_roots[dataset_id] = dataset_root
    shutil.rmtree(archive_root, ignore_errors=True)
    return counts, dataset_roots


def staged_session_uri(dataset_id, session_name):
    return f'{BEV_STAGE_ROOT_URI}/{dataset_id}/{session_name}'


def gcs_object_exists(path):
    uri = path_to_gs_uri(path)
    cmd = ['gcloud', 'storage', 'ls', uri]
    r = subprocess.run(cmd, cwd=str(ROOT), env=ENV, capture_output=True, text=True)
    return r.returncode == 0


def maybe_restore_train_checkpoint_from_drive():
    source_last = f'{DRIVE_TRAIN_OUT_URI}/last_model.pt'
    compat_last = f'{DRIVE_TRAIN_OUT_URI}/unet_last_model.pt'
    if (gcs_object_exists(source_last) or gcs_object_exists(compat_last)) and not (TRAIN_OUT / 'last_model.pt').exists() and not (TRAIN_OUT / 'unet_last_model.pt').exists():
        TRAIN_OUT.mkdir(parents=True, exist_ok=True)
        print(f'Restoring checkpoint via gcloud storage: {DRIVE_TRAIN_OUT_URI} -> {TRAIN_OUT}')
        gcloud_storage_rsync(DRIVE_TRAIN_OUT_URI, TRAIN_OUT)


def sync_train_checkpoint_to_drive():
    print(f'Syncing checkpoint via gcloud storage: {TRAIN_OUT} -> {DRIVE_TRAIN_OUT_URI}')
    gcloud_storage_rsync(TRAIN_OUT, DRIVE_TRAIN_OUT_URI)


print('BRANCH3_LABEL_BEV:', BRANCH3_LABEL_BEV)
print('BRANCH3_UNET_MODEL:', BRANCH3_UNET_MODEL)
print('TRAIN_SCRIPT:', TRAIN_SCRIPT)
print('Catalog URI:', GCS_CATALOG_URI)
print('Curated root:', CURATED_ROOT_URI)
print('BEV_STAGE_ROOT_URI:', BEV_STAGE_ROOT_URI)
print('Published root:', PUBLISHED_BRANCH3_ROOT_URI)
print('Work root :', ROOT)
print('Batched   :', UNET_TRAIN_IN_BATCHES)
print('Batch size:', UNET_SESSION_DOWNLOAD_BATCH_SIZE)


In [ ]:
# Branch 3 U-Net helpers and training wrappers.

def session_dirs(*roots):
    out = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        out.extend(sorted(p for p in root.iterdir() if p.is_dir() and p.name.startswith('session_')))
    return out


def generate_bev_labels_for_root(root, force=False):
    root = Path(root)
    if not root.exists() or not any(root.glob('session_*')):
        return
    cmd = [PY, BRANCH3_LABEL_BEV, '--root', root]
    if force:
        cmd.append('--force')
    run(cmd)


def build_unet_index(roots, out_dir, split='train', dataset_role='train'):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for session_dir in session_dirs(*roots):
        npz_paths = sorted(session_dir.glob('*_radar_tensors.npz'))
        label_dir = session_dir / 'bev_labels'
        meta_path = session_dir / 'bev_label_meta.json'
        if not npz_paths or not label_dir.exists() or not meta_path.exists():
            print(f'[SKIP] {session_dir.name}: missing radar_tensors or bev labels')
            continue
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        frame_ids = [int(x) for x in meta.get('frame_ids', [])]
        if not frame_ids:
            print(f'[SKIP] {session_dir.name}: no frame_ids')
            continue
        with np.load(str(npz_paths[0]), allow_pickle=False) as npz:
            rd_keys = set(k for k in npz.files if k.startswith('rd_'))
        n_session = 0
        for frame_num in frame_ids:
            # CSV radar_frame_num is one-based; tensor NPZ keys are rd_<zero-based frame_index>.
            rd_frame_index = int(frame_num) - 1
            label_path = label_dir / f'{int(frame_num):06d}.npy'
            if f'rd_{rd_frame_index}' not in rd_keys or not label_path.exists():
                continue
            rows.append({
                'session': session_dir.name,
                'session_dir': str(session_dir.resolve()),
                'radar_frame_num': int(frame_num),
                'rd_frame_index': rd_frame_index,
                'label_path': str(label_path.resolve()),
                'split': split,
            })
            n_session += 1
        print(f'{session_dir.name}: indexed {n_session} frames')
    if not rows:
        raise RuntimeError(f'No Branch 3 U-Net frames indexed for roots={roots}')
    df = pd.DataFrame(rows)
    out_csv = out_dir / 'unet_dataset_index.csv'
    df.to_csv(out_csv, index=False)
    stats = {
        'dataset_role': dataset_role,
        'total_frames': int(len(df)),
        'split_counts': {str(k): int(v) for k, v in df['split'].value_counts().to_dict().items()},
        'n_sessions': int(df['session'].nunique()),
        'sessions': sorted(df['session'].unique().tolist()),
    }
    (out_dir / 'unet_dataset_stats.json').write_text(json.dumps(stats, indent=2), encoding='utf-8')
    print('wrote index:', out_csv)
    print(json.dumps(stats, indent=2))
    return out_csv


def build_external_validation_dataset():
    final_csv = ROLLING_VAL_ROOT / 'dataset' / 'unet_dataset_index.csv'
    if final_csv.exists():
        print('Reusing external validation dataset:', final_csv)
        return final_csv
    val_records = validation_session_records()
    if not val_records:
        raise RuntimeError('Need at least one curated external-validation session for Branch 3 training.')
    clear_dir(ROLLING_VAL_ROOT)
    processing_root = ROLLING_VAL_ROOT / 'processing'
    counts, dataset_roots = download_session_records(val_records, processing_root)
    print('External validation download counts:', counts)
    for dataset_root in dataset_roots.values():
        generate_bev_labels_for_root(dataset_root)
    out_csv = build_unet_index(sorted(dataset_roots.values()), ROLLING_VAL_ROOT / 'dataset', split='val', dataset_role='external_val')
    print('Built external validation dataset:', out_csv)
    return out_csv


def train_unet(dataset_csv, external_val_csv=None, epochs=1, resume=True):
    TRAIN_OUT.mkdir(parents=True, exist_ok=True)
    cmd = [
        PY, TRAIN_SCRIPT,
        '--dataset', dataset_csv,
        '--out_dir', TRAIN_OUT,
        '--epochs', int(epochs),
        '--batch_size', int(UNET_BATCH_SIZE),
        '--lr', float(UNET_LR),
        '--weight_decay', float(UNET_WEIGHT_DECAY),
        '--base_ch', int(UNET_BASE_CH),
        '--num_workers', int(UNET_NUM_WORKERS),
        '--patience', int(UNET_PATIENCE if not UNET_TRAIN_IN_BATCHES else 0),
        '--best_metric', UNET_BEST_METRIC,
        '--seed', int(UNET_SEED),
    ]
    if external_val_csv is not None:
        cmd.extend(['--ext_val_dataset', external_val_csv])
    last_ckpt = TRAIN_OUT / 'last_model.pt'
    compat_last = TRAIN_OUT / 'unet_last_model.pt'
    if resume and last_ckpt.exists():
        cmd.extend(['--resume-checkpoint', last_ckpt])
    elif resume and compat_last.exists():
        cmd.extend(['--resume-checkpoint', compat_last])
    run(cmd)
    sync_train_checkpoint_to_drive()


def maybe_copy_best_to_runtime_path():
    best = TRAIN_OUT / 'best_model.pt'
    compat_best = TRAIN_OUT / 'unet_best_model.pt'
    source = best if best.exists() else compat_best
    if PROMOTE_BEST_TO_RUNTIME and source.exists():
        RUNTIME_UNET_PT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, RUNTIME_UNET_PT)
        print('Copied best Branch 3 U-Net to runtime path:', RUNTIME_UNET_PT)
        gcloud_storage_cp(RUNTIME_UNET_PT, PUBLISHED_RUNTIME_UNET_URI)
        print('Promoted runtime checkpoint to:', PUBLISHED_RUNTIME_UNET_URI)
    elif PROMOTE_BEST_TO_RUNTIME:
        print('No best model found to copy:', source)


## CPU/GPU staged workflow helpers

Run the CPU BEV-label stage on a CPU runtime, then switch to GPU and run the training stage. Both stages push/pull through GCS using `gcloud storage`.

In [ ]:
# CPU/GPU split functions for Branch 3 U-Net.

def _upload_labeled_sessions(local_root, stage_root_uri):
    local_root = Path(local_root)
    if not local_root.exists() or not any(local_root.glob('session_*')):
        print(f'[UPLOAD] no sessions under {local_root}; skipping')
        return 0
    print(f'[UPLOAD] labeled sessions: {local_root} -> {stage_root_uri}')
    gcloud_storage_rsync(local_root, stage_root_uri, delete=True)
    return len(list(local_root.glob('session_*')))


def run_cpu_build_bev_labeled_sessions(max_batches=None):
    ensure_expanded_sessions_available()
    all_train_records = session_records()
    batches = list(chunks(all_train_records, UNET_SESSION_DOWNLOAD_BATCH_SIZE))
    if max_batches is not None:
        batches = batches[:int(max_batches)]
    print(f'CPU BEV-label stage: {len(all_train_records)} train sessions; {len(batches)} batch(es) selected')

    val_records = validation_session_records()
    if val_records:
        val_stage = ROLLING_ROOT / 'cpu_stage_external_val'
        clear_dir(val_stage)
        counts, dataset_roots = download_session_records(val_records, val_stage / 'processing')
        print('[CPU STAGE] external validation download counts:', counts)
        for dataset_id, dataset_root in dataset_roots.items():
            generate_bev_labels_for_root(dataset_root)
            _upload_labeled_sessions(dataset_root, f'{BEV_STAGE_ROOT_URI}/{dataset_id}')
    else:
        print('[CPU STAGE] no external validation records found')

    manifest = {
        'stage': 'cpu_build_bev_labeled_sessions',
        'train_stage_root': BEV_STAGE_ROOT_URI,
        'created_unix': int(time.time()),
        'batches': [],
    }

    for batch_idx, batch_records in batches:
        batch_root = ROLLING_ROOT / 'cpu_stage_train' / f'batch_{batch_idx:04d}'
        clear_dir(batch_root)
        counts, dataset_roots = download_session_records(batch_records, batch_root / 'processing')
        print(f'[CPU STAGE] batch {batch_idx}: downloaded {counts}')
        for dataset_id, dataset_root in dataset_roots.items():
            generate_bev_labels_for_root(dataset_root)
            _upload_labeled_sessions(dataset_root, f'{BEV_STAGE_ROOT_URI}/{dataset_id}')
        manifest['batches'].append({
            'batch_idx': int(batch_idx),
            'counts': counts,
            'sessions': [name for _dataset_id, name, _uri, _split in batch_records],
        })
        clear_dir(batch_root)
        run_shell('df -h /content /content/work || df -h .')

    manifest_path = ROLLING_ROOT / 'cpu_bev_stage_manifest.json'
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    run(['gcloud', 'storage', 'cp', manifest_path, f'{BEV_STAGE_ROOT_URI}/cpu_bev_stage_manifest.json'])
    print('CPU BEV-label stage complete:', BEV_STAGE_ROOT_URI)
    return manifest


def staged_train_records():
    return [
        (dataset_id, session_name, staged_session_uri(dataset_id, session_name), split)
        for dataset_id, session_name, _processed_uri, split in session_records()
    ]


def build_staged_external_validation_dataset():
    final_csv = ROLLING_VAL_ROOT / 'dataset' / 'unet_dataset_index.csv'
    if final_csv.exists():
        print('Reusing local external validation dataset:', final_csv)
        return final_csv
    clear_dir(ROLLING_VAL_ROOT)
    records = [
        (dataset_id, session_name, staged_session_uri(dataset_id, session_name), split)
        for dataset_id, session_name, _processed_uri, split in validation_session_records()
    ]
    counts, dataset_roots = download_staged_session_records(records, ROLLING_VAL_ROOT / 'processing')
    print('Staged external validation download counts:', counts)
    out_csv = build_unet_index(sorted(dataset_roots.values()), ROLLING_VAL_ROOT / 'dataset', split='val', dataset_role='external_val')
    print('Built external validation dataset:', out_csv)
    return out_csv


def download_staged_session_records(records, processing_root):
    processing_root = Path(processing_root)
    processing_root.mkdir(parents=True, exist_ok=True)
    counts = {}
    dataset_roots = {}
    for dataset_id, session_name, source_uri, _split in records:
        dataset_root = processing_root / dataset_id
        dataset_root.mkdir(parents=True, exist_ok=True)
        dst = dataset_root / session_name
        shutil.rmtree(dst, ignore_errors=True)
        print(f'Downloading staged labeled session via gcloud: {source_uri} -> {dst}')
        gcloud_storage_rsync(source_uri, dst)
        counts[dataset_id] = counts.get(dataset_id, 0) + 1
        dataset_roots[dataset_id] = dataset_root
    return counts, dataset_roots


def run_gpu_train_unet_from_staged_labels(max_batches=None):
    maybe_restore_train_checkpoint_from_drive()
    external_val_csv = build_staged_external_validation_dataset()
    records = staged_train_records()

    if UNET_TRAIN_IN_BATCHES:
        batches = list(chunks(records, UNET_SESSION_DOWNLOAD_BATCH_SIZE))
        if max_batches is not None:
            batches = batches[:int(max_batches)]
        print(f'GPU U-Net training from staged labels: {len(records)} sessions; {len(batches)} batch(es)')
        print(f'Streaming passes: {UNET_STREAMING_PASSES}')
        print(f'Epochs per loaded batch: {UNET_EPOCHS_PER_SESSION_BATCH}')
        for pass_idx in range(int(UNET_STREAMING_PASSES)):
            print(f'===== Branch 3 GPU streaming pass {pass_idx + 1}/{UNET_STREAMING_PASSES} =====')
            for batch_idx, batch_records in batches:
                global_batch_id = pass_idx * len(batches) + batch_idx
                batch_root = ROLLING_ROOT / f'gpu_train_batch_{global_batch_id:04d}'
                clear_dir(batch_root)
                counts, dataset_roots = download_staged_session_records(batch_records, batch_root / 'processing')
                print(f'GPU training batch {global_batch_id}: downloaded {counts}')
                dataset_csv = build_unet_index(
                    sorted(dataset_roots.values()),
                    ROLLING_DATASET_ROOT / f'gpu_batch_{global_batch_id:04d}',
                    split='train',
                    dataset_role='rolling_train',
                )
                train_unet(dataset_csv, external_val_csv=external_val_csv, epochs=UNET_EPOCHS_PER_SESSION_BATCH, resume=True)
                clear_dir(batch_root)
                run_shell('df -h /content /content/work || df -h .')
    else:
        full_processing = FULL_ROOT / 'gpu_processing'
        clear_dir(full_processing)
        counts, dataset_roots = download_staged_session_records(records, full_processing)
        print(f'GPU full training downloaded {counts}')
        dataset_csv = build_unet_index(
            sorted(dataset_roots.values()),
            ROLLING_DATASET_ROOT / 'gpu_full_train',
            split='train',
            dataset_role='full_train',
        )
        train_unet(dataset_csv, external_val_csv=external_val_csv, epochs=UNET_FULL_EPOCHS, resume=True)

    maybe_copy_best_to_runtime_path()
    sync_train_checkpoint_to_drive()
    print('Branch 3 U-Net GPU retrain complete. Output:', TRAIN_OUT)
    return TRAIN_OUT


## Run stages

Recommended:

1. CPU runtime: `run_cpu_build_bev_labeled_sessions(max_batches=1)` for smoke, then full `run_cpu_build_bev_labeled_sessions()`.
2. GPU runtime: rerun setup/helper cells, then `run_gpu_train_unet_from_staged_labels(max_batches=1)` for smoke or full `run_gpu_train_unet_from_staged_labels()`.

The old one-shot flow has been replaced so BEV label generation is not repeated on the GPU runtime.

In [ ]:
# CPU runtime: generate BEV labels and upload labeled session folders.
# Smoke:
# run_cpu_build_bev_labeled_sessions(max_batches=1)

# Full:
# run_cpu_build_bev_labeled_sessions()


In [ ]:
# GPU runtime: train from already-labeled staged sessions.
# Smoke:
# run_gpu_train_unet_from_staged_labels(max_batches=1)

# Full:
# run_gpu_train_unet_from_staged_labels()


In [ ]:
# Quick checkpoint validation and final copy summary.
best = TRAIN_OUT / 'unet_best_model.pt'
last = TRAIN_OUT / 'unet_last_model.pt'
print('TRAIN_OUT:', TRAIN_OUT)
print('best exists:', best.exists(), best)
print('last exists:', last.exists(), last)
print('runtime exists:', RUNTIME_UNET_PT.exists(), RUNTIME_UNET_PT)

if best.exists():
    import torch
    ckpt = torch.load(best, map_location='cpu', weights_only=False)
    print('best checkpoint keys:', sorted(k for k in ckpt.keys() if k != 'model_state'))
    print('epoch:', ckpt.get('epoch'), 'in_channels:', ckpt.get('in_channels'), 'out:', (ckpt.get('out_h'), ckpt.get('out_w')))

if DRIVE_TRAIN_OUT.exists():
    print('Drive output:', DRIVE_TRAIN_OUT)
